# Pre-Training Table

This notebook builds a **pre-training table** from the master join.

Each event in the master join is already mapped to a specific tracking frame (its nearest frame by timestamp). That tracking frame's `t.Videotimestamp` is used directly as the **source of truth** for the event's location in time — no new timestamp column is created.

**Rule**: every tracking frame whose `t.Videotimestamp` falls within ±1 second of a real event's `t.Videotimestamp` is labeled with that event type. Frames outside every event window receive `"no event"`. `"no event"` rows from the master join are never used as event anchors — they do not expand.

When a frame falls within 1 second of multiple events it is labeled with the nearest one.

New columns created in this step (`p.*`):

| Column | Description |
|---|---|
| `p.actual_event_frame` | `t.Videotimestamp` of the nearest event anchor assigned to this frame (`NaN` when no event) |
| `p.event_label` | Event type from that anchor if within ±1 s, otherwise `"no event"` |
| `p.dist_to_actual_event` | Absolute distance in seconds to the event anchor (`NaN` when no event) |

All original tracking columns (`t.*`) are preserved unchanged. All `e.*` event columns are dropped.

## 1. Setup

In [74]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

MODEL_BASE_DIR = PROJECT_ROOT / "data" / "processed" / "model_base"
MASTER_JOIN_PATH = MODEL_BASE_DIR / "master_join_table.parquet"
OUTPUT_PATH = MODEL_BASE_DIR / "pre_training_table.parquet"

WINDOW_SEC = 1.0  # ±1 second around each event frame
FPS = 10.0        # tracking frame rate

assert MASTER_JOIN_PATH.exists(), f"Missing {MASTER_JOIN_PATH}. Run: python main.py step2"
print("Master join:", MASTER_JOIN_PATH)

Master join: /Users/nataliaurrea/Documents/IE/MBDS/Term III/Driblab/data/processed/model_base/master_join_table.parquet


## 2. Load master join

In [75]:
master_join = pd.read_parquet(MASTER_JOIN_PATH)

n_event_rows = int((master_join["e.event.event_type_name"] != "no event").sum())
print(f"Total tracking frames : {len(master_join):,}")
print(f"Frames with event (current, 1-per-event) : {n_event_rows:,}")
print(f"Frames without event                      : {len(master_join) - n_event_rows:,}")
print()
print("Event type breakdown (master join):")
display(
    master_join["e.event.event_type_name"]
    .value_counts()
    .drop("no event", errors="ignore")
    .rename("frames")
    .to_frame()
)

Total tracking frames : 1,986,630
Frames with event (current, 1-per-event) : 42,437
Frames without event                      : 1,944,193

Event type breakdown (master join):


,frames
e.event.event_type_name,
PASS,31670
BALL TOUCH,4463
TACKLE,1647
AERIAL,1496
BALL RECOVERY,1325
TAKEON,1141
FOUL,695


## 3. Verify `t.Videotimestamp`

`t.Videotimestamp` is already stored by the tracking provider as a sub-second float at 10 Hz: `1.0, 1.1, 1.2, … 1.9, 2.0, 2.1, …` — one unique value per frame. No new column is needed; `t.Videotimestamp` is used directly throughout this notebook as the timestamp for all ±1 s distance calculations.

`t.match_clock` only has whole-second resolution (10 frames share the same `[min, sec]` value) and is not used here.

In [76]:
# Safety cast: parquet preserves float type, but ensure numeric in case of string serialisation
master_join["t.Videotimestamp"] = pd.to_numeric(master_join["t.Videotimestamp"], errors="coerce")

print("t.Videotimestamp — one unique sub-second value per frame at 10 Hz:")
display(
    master_join[
        ["t.match_id", "t.period", "t.frame", "t.match_clock", "t.Videotimestamp"]
    ].head(12)
)

t.Videotimestamp — one unique sub-second value per frame at 10 Hz:


,t.match_id,t.period,t.frame,t.match_clock,t.Videotimestamp
0,678949,1,0,"[0,1]",1.0
1,678949,1,1,"[0,1]",1.1
2,678949,1,2,"[0,1]",1.2
3,678949,1,3,"[0,1]",1.3
4,678949,1,4,"[0,1]",1.4
5,678949,1,5,"[0,1]",1.5
6,678949,1,6,"[0,1]",1.6
7,678949,1,7,"[0,1]",1.7
8,678949,1,8,"[0,1]",1.8
9,678949,1,9,"[0,1]",1.9


## 4. Apply ±1 second event window

For each `(match, period)` group, `pd.merge_asof(direction="nearest")` finds — for every tracking frame — the single nearest event frame by `p.tracking_sec`. If the distance is ≤ `WINDOW_SEC = 1.0 s`, the frame is labeled with that event type; otherwise it receives `"no event"`.

**Important: `"no event"` rows are never event anchors.**  
The event anchor table is built exclusively from rows where `e.event.event_type_name != "no event"`. A frame that was `"no event"` in the master join does not propagate that label to neighbouring frames — it is either overwritten by a nearby real event, or remains `"no event"` if no real event falls within ±1 s.

**Overlap handling** is explained in the markdown before section 6.

In [77]:
# Build event anchor table from real events only — "no event" rows are excluded entirely.
# p.actual_event_frame stores the t.Videotimestamp of the event's own tracking frame.
events_df = master_join.loc[
    master_join["e.event.event_type_name"] != "no event",
    ["t.match_id", "t.period", "t.Videotimestamp", "e.event.event_type_name"],
].rename(columns={
    "t.Videotimestamp": "p.actual_event_frame",
    "e.event.event_type_name": "p.nearest_event_label",
})

labeled_chunks = []

for (match_id, period), period_tracking in master_join.groupby(
    ["t.match_id", "t.period"], sort=False
):
    # Isolate events for this match + period, sorted for merge_asof
    period_events = (
        events_df[
            (events_df["t.match_id"] == match_id)
            & (events_df["t.period"] == period)
        ]
        .sort_values("p.actual_event_frame")
    )

    # Sort tracking frames by t.Videotimestamp — required by merge_asof
    chunk = period_tracking.sort_values("t.Videotimestamp").copy()

    if period_events.empty:
        # No real events in this period → every frame stays "no event"
        chunk["p.event_label"] = "no event"
        chunk["p.dist_to_actual_event"] = np.nan
        chunk["p.actual_event_frame"] = np.nan
        labeled_chunks.append(chunk)
        continue

    # For each tracking frame, find the single nearest event frame.
    # direction="nearest" always returns the closest anchor regardless of distance;
    # we then apply the WINDOW_SEC threshold ourselves.
    merged = pd.merge_asof(
        chunk,
        period_events[["p.actual_event_frame", "p.nearest_event_label"]],
        left_on="t.Videotimestamp",
        right_on="p.actual_event_frame",
        direction="nearest",
    )

    # p.dist_to_actual_event: absolute seconds between this frame and its nearest event anchor
    dist = (merged["t.Videotimestamp"] - merged["p.actual_event_frame"]).abs()
    within = dist <= WINDOW_SEC

    # p.event_label: event type if within window, "no event" otherwise
    merged["p.event_label"] = np.where(
        within, merged["p.nearest_event_label"], "no event"
    )
    # p.dist_to_actual_event: distance to anchor for labeled frames, NaN for "no event" frames
    merged["p.dist_to_actual_event"] = np.where(within, dist, np.nan)
    # Clear the anchor timestamp for frames that fell outside the window
    merged.loc[~within, "p.actual_event_frame"] = np.nan

    # Drop the temporary label column used only during the merge
    merged = merged.drop(columns=["p.nearest_event_label"])
    labeled_chunks.append(merged)

pre_training = pd.concat(labeled_chunks, ignore_index=True)

# Drop all e.* columns — only t.* tracking columns and p.* derived columns are kept
e_cols = [c for c in pre_training.columns if c.startswith("e.")]
pre_training = pre_training.drop(columns=e_cols)

print(f"Pre-training table: {len(pre_training):,} rows")
print(f"Columns: {pre_training.shape[1]}  (e.* dropped: {len(e_cols)})")

Pre-training table: 1,986,630 rows
Columns: 125  (e.* dropped: 24)


### Label columns — how they are calculated

**`p.actual_event_frame`**  
The `t.Videotimestamp` of the real event's own tracking frame — the frame that the master join originally matched to the event. This is the anchor timestamp used to define the ±1 s window. It is `NaN` for any frame labeled `"no event"`.  
Source: taken from `t.Videotimestamp` of master join rows where `e.event.event_type_name != "no event"`.

**`p.event_label`**  
The event type assigned to this tracking frame.  
Rule: if `|t.Videotimestamp − p.actual_event_frame| ≤ WINDOW_SEC`, the frame receives the event type; otherwise `"no event"`.  
Source: copied from `e.event.event_type_name` of the nearest event anchor.  
Note: `"no event"` rows in the master join are **never** anchors, so this label can only be a real event type or `"no event"` due to distance.

**`p.dist_to_actual_event`**  
The absolute time distance between this frame and its assigned event anchor.  
Formula: `|t.Videotimestamp − p.actual_event_frame|`  
Range: `0.0` (the exact event frame itself) to `WINDOW_SEC` (the farthest labeled frame). `NaN` for `"no event"` frames.

## 5. Label expansion statistics

In [78]:
# Compare labeled frame counts before (master join, 1 frame per event) and after (±1 s window)
# "no event" is excluded from this comparison — it is not an event type being expanded,
# it is simply the label for frames that fall outside every event window.
old_counts = (
    master_join["e.event.event_type_name"]
    .value_counts()
    .drop("no event", errors="ignore")   # exclude "no event" — not an event anchor
    .rename("master_join_frames")
)
new_counts = (
    pre_training["p.event_label"]
    .value_counts()
    .drop("no event", errors="ignore")   # exclude "no event" for the same reason
    .rename("pre_training_frames")
)

comparison = (
    old_counts.to_frame()
    .join(new_counts, how="outer")
    .fillna(0)
    .astype(int)
)
comparison["expansion_factor"] = (
    comparison["pre_training_frames"] / comparison["master_join_frames"].replace(0, np.nan)
).round(1)

# How many frames moved from "no event" to a real label
no_event_before = int((master_join["e.event.event_type_name"] == "no event").sum())
no_event_after = int((pre_training["p.event_label"] == "no event").sum())

print(f"Frames still labeled 'no event': {no_event_after:,}  (was {no_event_before:,} in master join)")
print(f"Frames converted from 'no event' to a real event label: {no_event_before - no_event_after:,}")
print()
print("Expansion of real event labels:")
display(comparison.sort_values("pre_training_frames", ascending=False))

Frames still labeled 'no event': 1,240,461  (was 1,944,193 in master join)
Frames converted from 'no event' to a real event label: 703,732

Expansion of real event labels:


,master_join_frames,pre_training_frames,expansion_factor
e.event.event_type_name,,,
PASS,31670,576467,18.2
BALL TOUCH,4463,74355,16.7
TACKLE,1647,25445,15.4
BALL RECOVERY,1325,19013,14.3
AERIAL,1496,18497,12.4
TAKEON,1141,18046,15.8
FOUL,695,14346,20.6


## 6. Overlap logic — what happens when two event windows collide

Two events whose anchor frames are less than `2 × WINDOW_SEC = 2.0 s` apart will have overlapping ±1 s windows. Any tracking frame inside that overlap zone is within 1 second of **both** events.

**How the overlap is resolved:** `pd.merge_asof(direction="nearest")` assigns each tracking frame to the single event whose anchor timestamp is closest to the frame's `p.tracking_sec`. The further event is never considered for that frame.

**Example:**  
- Event A (PASS) anchored at `t = 45.2 s`  
- Event B (BALL TOUCH) anchored at `t = 46.0 s` (gap = 0.8 s → windows overlap by 1.2 s)  
- Overlap zone: frames between `t = 45.0 s` and `t = 46.0 s` (the intersection of [44.2, 46.2] and [45.0, 47.0])  

For a frame at `t = 45.5 s`:  
- Distance to A = |45.5 − 45.2| = 0.3 s  
- Distance to B = |45.5 − 46.0| = 0.5 s  
- **Result: labeled PASS** (A is closer, and 0.3 s ≤ 1.0 s)  

For a frame at `t = 45.7 s`:  
- Distance to A = 0.5 s, distance to B = 0.3 s  
- **Result: labeled BALL TOUCH** (B is closer)  

The **midpoint** between two event anchors is the exact split boundary. Frames before the midpoint go to the earlier event; frames at or after the midpoint go to the later event. In the case of a perfect tie (frame exactly equidistant) `merge_asof` picks the event that appears first in the sorted anchor list (i.e., the earlier-anchored one).

The preview below shows 1.5 s either side of the first PASS event — if the window is open at the edge it means no other event is within 1 s on that side.

In [84]:
# Find the first PASS-labeled frame to use as the preview anchor
first_pass = pre_training[pre_training["p.event_label"] == "PASS"].iloc[100]

# Show ±2 s around that event so we can see the full ±2 s labeled window
# plus the "no event" frames just outside it
window_preview = pre_training[
    (pre_training["t.match_id"] == first_pass["t.match_id"])
    & (pre_training["t.period"] == first_pass["t.period"])
    & (pre_training["t.Videotimestamp"] >= first_pass["p.actual_event_frame"] - 2)
    & (pre_training["t.Videotimestamp"] <= first_pass["p.actual_event_frame"] + 2)
].sort_values("t.Videotimestamp")[
    [
        "t.match_id", "t.period", "t.frame",
        "t.Videotimestamp", "p.actual_event_frame",
        "p.event_label", "p.dist_to_actual_event",
        "t.ball_x", "t.ball_y",
    ]
]

print(
    f"Frames ±2 s around PASS anchor at p.actual_event_frame={first_pass['p.actual_event_frame']:.3f}s "
    f"(match {first_pass['t.match_id']}, period {first_pass['t.period']}):"
)
display(window_preview)

Frames ±2 s around PASS anchor at p.actual_event_frame=39.420s (match 678949, period 1):


,t.match_id,t.period,t.frame,t.Videotimestamp,p.actual_event_frame,p.event_label,p.dist_to_actual_event,t.ball_x,t.ball_y
368,678949,1,368,37.48,37.28,PASS,0.20,NaN,NaN
369,678949,1,369,37.58,37.78,BALL TOUCH,0.20,NaN,NaN
370,678949,1,370,37.68,37.78,BALL TOUCH,0.10,NaN,NaN
371,678949,1,371,37.78,37.78,BALL TOUCH,0.00,NaN,NaN
372,678949,1,372,37.88,37.78,BALL TOUCH,0.10,NaN,NaN
373,678949,1,373,37.98,37.78,BALL TOUCH,0.20,NaN,NaN
374,678949,1,374,38.08,37.78,BALL TOUCH,0.30,NaN,NaN
375,678949,1,375,38.18,37.78,BALL TOUCH,0.40,NaN,NaN
376,678949,1,376,38.28,37.78,BALL TOUCH,0.50,NaN,NaN
377,678949,1,377,38.38,37.78,BALL TOUCH,0.60,NaN,NaN


## 7. Save pre-training table

Output contains all `t.*` tracking columns plus the three `p.*` columns created in this step: `p.actual_event_frame`, `p.event_label`, and `p.dist_to_actual_event`. All `e.*` columns from the master join are already dropped from `pre_training`.

In [81]:
tracking_cols = [c for c in pre_training.columns if c.startswith("t.")]
p_cols = ["p.actual_event_frame", "p.event_label", "p.dist_to_actual_event"]

output_df = pre_training[tracking_cols + p_cols].copy()

output_df.to_parquet(OUTPUT_PATH, index=False)

print(f"Saved {len(output_df):,} rows → {OUTPUT_PATH}")
print(f"Columns: {len(output_df.columns)}  (t.*: {len(tracking_cols)}, p.*: {len(p_cols)})")
print()
print("Label distribution:")
display(
    output_df["p.event_label"]
    .value_counts()
    .rename("frames")
    .to_frame()
    .assign(pct=lambda x: (x["frames"] / len(output_df) * 100).round(2))
)

Saved 1,986,630 rows → /Users/nataliaurrea/Documents/IE/MBDS/Term III/Driblab/data/processed/model_base/pre_training_table.parquet
Columns: 124  (t.*: 121, p.*: 3)

Label distribution:


,frames,pct
p.event_label,,
no event,1240461,62.44
PASS,576467,29.02
BALL TOUCH,74355,3.74
TACKLE,25445,1.28
BALL RECOVERY,19013,0.96
AERIAL,18497,0.93
TAKEON,18046,0.91
FOUL,14346,0.72


In [82]:
# Preview the pre_training table
print(f"pre_training: {len(pre_training):,} rows x {pre_training.shape[1]} columns\n")

print("Column names:")
print(pre_training.columns.tolist(), "\n")

print("Dtypes:")
print(pre_training.dtypes, "\n")

print("Head (10 rows):")
display(pre_training.head(10))

print("Tail (10 rows):")
display(pre_training.tail(10))

pre_training: 1,986,630 rows x 125 columns

Column names:
['t.match_id', 't.match_clock', 't.frame', 't.Videotimestamp', 't.period', 't.cam', 't.ball_x', 't.ball_y', 't.ball_z', 't.player_count', 't.visible_player_count', 't.player_01_team_id', 't.player_01_id', 't.player_01_x', 't.player_01_y', 't.player_01_visible', 't.player_02_team_id', 't.player_02_id', 't.player_02_x', 't.player_02_y', 't.player_02_visible', 't.player_03_team_id', 't.player_03_id', 't.player_03_x', 't.player_03_y', 't.player_03_visible', 't.player_04_team_id', 't.player_04_id', 't.player_04_x', 't.player_04_y', 't.player_04_visible', 't.player_05_team_id', 't.player_05_id', 't.player_05_x', 't.player_05_y', 't.player_05_visible', 't.player_06_team_id', 't.player_06_id', 't.player_06_x', 't.player_06_y', 't.player_06_visible', 't.player_07_team_id', 't.player_07_id', 't.player_07_x', 't.player_07_y', 't.player_07_visible', 't.player_08_team_id', 't.player_08_id', 't.player_08_x', 't.player_08_y', 't.player_08_visi

,t.match_id,t.match_clock,t.frame,t.Videotimestamp,t.period,t.cam,t.ball_x,t.ball_y,t.ball_z,t.player_count,t.visible_player_count,t.player_01_team_id,t.player_01_id,t.player_01_x,t.player_01_y,t.player_01_visible,t.player_02_team_id,t.player_02_id,t.player_02_x,t.player_02_y,t.player_02_visible,t.player_03_team_id,t.player_03_id,t.player_03_x,t.player_03_y,t.player_03_visible,t.player_04_team_id,t.player_04_id,t.player_04_x,t.player_04_y,t.player_04_visible,t.player_05_team_id,t.player_05_id,t.player_05_x,t.player_05_y,t.player_05_visible,t.player_06_team_id,t.player_06_id,t.player_06_x,t.player_06_y,t.player_06_visible,t.player_07_team_id,t.player_07_id,t.player_07_x,t.player_07_y,t.player_07_visible,t.player_08_team_id,t.player_08_id,t.player_08_x,t.player_08_y,t.player_08_visible,t.player_09_team_id,t.player_09_id,t.player_09_x,t.player_09_y,t.player_09_visible,t.player_10_team_id,t.player_10_id,t.player_10_x,t.player_10_y,t.player_10_visible,t.player_11_team_id,t.player_11_id,t.player_11_x,t.player_11_y,t.player_11_visible,t.player_12_team_id,t.player_12_id,t.player_12_x,t.player_12_y,t.player_12_visible,t.player_13_team_id,t.player_13_id,t.player_13_x,t.player_13_y,t.player_13_visible,t.player_14_team_id,t.player_14_id,t.player_14_x,t.player_14_y,t.player_14_visible,t.player_15_team_id,t.player_15_id,t.player_15_x,t.player_15_y,t.player_15_visible,t.player_16_team_id,t.player_16_id,t.player_16_x,t.player_16_y,t.player_16_visible,t.player_17_team_id,t.player_17_id,t.player_17_x,t.player_17_y,t.player_17_visible,t.player_18_team_id,t.player_18_id,t.player_18_x,t.player_18_y,t.player_18_visible,t.player_19_team_id,t.player_19_id,t.player_19_x,t.player_19_y,t.player_19_visible,t.player_20_team_id,t.player_20_id,t.player_20_x,t.player_20_y,t.player_20_visible,t.player_21_team_id,t.player_21_id,t.player_21_x,t.player_21_y,t.player_21_visible,t.player_22_team_id,t.player_22_id,t.player_22_x,t.player_22_y,t.player_22_visible,nearest_timestamp_distance_sec,p.actual_event_frame,p.event_label,p.dist_to_actual_event
0,678949,"[0,1]",0,1.0,1,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,1.2,PASS,0.2
1,678949,"[0,1]",1,1.1,1,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,1.2,PASS,0.1
2,678949,"[0,1]",2,1.2,1,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,0.011,1.2,PASS,0.0
3,678949,"[0,1]",3,1.3,1,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None

Tail (10 rows):


,t.match_id,t.match_clock,t.frame,t.Videotimestamp,t.period,t.cam,t.ball_x,t.ball_y,t.ball_z,t.player_count,t.visible_player_count,t.player_01_team_id,t.player_01_id,t.player_01_x,t.player_01_y,t.player_01_visible,t.player_02_team_id,t.player_02_id,t.player_02_x,t.player_02_y,t.player_02_visible,t.player_03_team_id,t.player_03_id,t.player_03_x,t.player_03_y,t.player_03_visible,t.player_04_team_id,t.player_04_id,t.player_04_x,t.player_04_y,t.player_04_visible,t.player_05_team_id,t.player_05_id,t.player_05_x,t.player_05_y,t.player_05_visible,t.player_06_team_id,t.player_06_id,t.player_06_x,t.player_06_y,t.player_06_visible,t.player_07_team_id,t.player_07_id,t.player_07_x,t.player_07_y,t.player_07_visible,t.player_08_team_id,t.player_08_id,t.player_08_x,t.player_08_y,t.player_08_visible,t.player_09_team_id,t.player_09_id,t.player_09_x,t.player_09_y,t.player_09_visible,t.player_10_team_id,t.player_10_id,t.player_10_x,t.player_10_y,t.player_10_visible,t.player_11_team_id,t.player_11_id,t.player_11_x,t.player_11_y,t.player_11_visible,t.player_12_team_id,t.player_12_id,t.player_12_x,t.player_12_y,t.player_12_visible,t.player_13_team_id,t.player_13_id,t.player_13_x,t.player_13_y,t.player_13_visible,t.player_14_team_id,t.player_14_id,t.player_14_x,t.player_14_y,t.player_14_visible,t.player_15_team_id,t.player_15_id,t.player_15_x,t.player_15_y,t.player_15_visible,t.player_16_team_id,t.player_16_id,t.player_16_x,t.player_16_y,t.player_16_visible,t.player_17_team_id,t.player_17_id,t.player_17_x,t.player_17_y,t.player_17_visible,t.player_18_team_id,t.player_18_id,t.player_18_x,t.player_18_y,t.player_18_visible,t.player_19_team_id,t.player_19_id,t.player_19_x,t.player_19_y,t.player_19_visible,t.player_20_team_id,t.player_20_id,t.player_20_x,t.player_20_y,t.player_20_visible,t.player_21_team_id,t.player_21_id,t.player_21_x,t.player_21_y,t.player_21_visible,t.player_22_team_id,t.player_22_id,t.player_22_x,t.player_22_y,t.player_22_visible,nearest_timestamp_distance_sec,p.actual_event_frame,p.event_label,p.dist_to_actual_event
1986620,745399,"[94,28]",57060,5668.0,2,"[[11.0,-395.73],[6158.83,-5488.59],[50.18,70.6...",50.54,55.35,0.000,22,18,10,1302799.0,58.72,18.57,True,10,1323295.0,70.22,26.21,True,10,1533093.0,76.13,64.87,True,10,69813.0,70.55,50.34,True,10,1510226.0,79.14,35.53,True,10,1646676.0,51.64,37.02,True,10,1329892.0,68.13,5.60,False,10,1474405.0,19.71,35.09,False,10,189161.0,56.49,30.16,True,10,1217662.0,61.45,43.11,True,10,1323007.0,73.45,56.26,True,3664,1334145.0,77.65,18.50,False,3664,241417.0,75.47,55.32,True,3664,1162897.0,98.37,34.05,False,3664,1157019.0,62.21,41.47,True,3664,1691049.0,67.59,35.18,True,3664,1158191.0,75.27,28.36,True,3664,1156797.0,77.42,43.85,True,3664,1296848.0,67.30,21.35,True,3664,1797970.0,78.54,36.25,True,3664,1322735.0,66.93,52.55,True,3664,1294971.0,57.99,36.70,True,NaN,NaN,no event,NaN
1986621,745399,"[94,28]",57061,5668.1,2,"[[128.04,810.94],[2225.32,-1898.95],[50.11,70....",50.68,55.82,0.000,22,18,10,1302799.0,58.58,18.20,True,10,1323295.0,70.14,25.76,True,10,1533093.0,76.12,64.78,True,10,69813.0,70.29,50.32,True,10,1510226.0,79.13,35.38,True,10,1646676.0,51.66,36.97,True,10,1329892.0,68.24,5.71,False,10,1474405.0,19.76,35.15,False,10,189161.0,56.20,29.88,True,10,1217662.0,61.58,43.02,True,10,1323007.0,73.44,56.22,True,3664,1334145.0,77.68,18.11,False,3664,241417.0,75.38,55.22,True,3664,1162897.0,98.19,34.06,False,3664,1157019.0,62.28,41.23,True,3664,1691049.0,67.53,34.93,True,3664,1158191.0,75.14,27.99,True,3664,1156797.0,77.38,43.72,True,3664,1296848.0,67.13,21.04,True,3664,1797970.0,78.39,36.10,True,3664,1322735.0,66.87,52.38,True,3664,1294971.0,57.76,36.43,True,NaN,NaN,no event,NaN
1986622,745399,"[94,28]",57062,5668.2,2,"[[245.08,2017.6],[-1708.2,1690.69],[50.03,69.8...",50.81,56.29,0.000,22,18,10,1302799.0,58.45,17.84,True,10,1323295.0,70.06,25.32,True,10,1533093.0,76.11,64.69,True,10,69813.0,70.03,50.30,True,10,1510226.0,79.13,35.22,True,10,1646676.0,51.69,36.91,True,10,1329892.0,68.36,5.82,

## 8. Frame timing audit — are `t.Videotimestamp` gaps uniform?

`t.Videotimestamp` is the raw value from the tracking provider and is stored unchanged by the ETL. It is **not** reconstructed — unlike the internal `_tracking_match_clock_seconds` which is built as `whole_second + cumcount / fps`.

Gaps that deviate from 0.10 s in `p.event_dist_sec` reflect genuine non-uniformity in the provider's timestamps. Two likely causes:

1. **Dropped frame** — the provider skipped one frame, leaving a 0.20 s gap.
2. **25fps source video** — tracking derived from 25fps video sampled every ~2.5 frames alternates between 0.08 s and 0.12 s gaps to average 10 Hz.

This does **not** break the ±1 s labeling: both sides of the distance subtraction (`t.Videotimestamp` of each frame and `p.event_frame_sec` of the anchor) come from the same raw values, so distances are accurate.

In [83]:
# Compute frame-to-frame t.Videotimestamp gaps per (match, period)
# Expected: all gaps = 0.1 s. Any other value = irregular frame timing in raw data.
audit_rows = []
for (mid, per), grp in pre_training.groupby(["t.match_id", "t.period"]):
    ts = grp.sort_values("t.Videotimestamp")["t.Videotimestamp"].values
    gaps = np.diff(ts).round(4)  # round to 4dp to absorb float noise
    unique_gaps, counts = np.unique(gaps, return_counts=True)
    n_irregular = int((~np.isclose(gaps, 0.1, atol=1e-3)).sum())
    irregular_vals = sorted(set(gaps[~np.isclose(gaps, 0.1, atol=1e-3)].round(3)))
    audit_rows.append({
        "match_id": mid,
        "period": per,
        "total_frames": len(ts),
        "n_gaps": len(gaps),
        "n_irregular": n_irregular,
        "pct_irregular": round(n_irregular / len(gaps) * 100, 2) if len(gaps) else 0,
        "irregular_gap_values": irregular_vals,
    })

audit_df = pd.DataFrame(audit_rows)
print(f"Periods with ALL uniform 0.1 s gaps : {(audit_df['n_irregular'] == 0).sum()} / {len(audit_df)}")
print(f"Periods with ANY irregular gap      : {(audit_df['n_irregular'] > 0).sum()} / {len(audit_df)}")
print(f"Total irregular gaps across dataset : {audit_df['n_irregular'].sum():,}")
print()
print("Breakdown per (match, period):")
display(
    audit_df[["match_id", "period", "total_frames", "n_irregular", "pct_irregular", "irregular_gap_values"]]
    .sort_values("n_irregular", ascending=False)
)

Periods with ALL uniform 0.1 s gaps : 0 / 66
Periods with ANY irregular gap      : 66 / 66
Total irregular gaps across dataset : 17,417

Breakdown per (match, period):


,match_id,period,total_frames,n_irregular,pct_irregular,irregular_gap_values
65,745399,2,29044,408,1.40,"[0.02, 0.08]"
42,684147,1,29054,391,1.35,"[0.02, 0.08, 0.12]"
43,684147,2,30084,386,1.28,"[0.02, 0.08, 1.1]"
44,689340,1,28536,377,1.32,"[0.02, 0.08, 0.62, 1.1]"
21,682815,2,32172,374,1.16,"[0.02, 0.08, 1.1, 1.6, 5.1]"
...,...,...,...,...,...,...
10,679088,1,31262,179,0.57,"[0.02, 0.04, 0.06, 0.08, 0.12, 0.6]"
28,683261,1,30784,179,0.58,"[0.02, 0.04, 0.06, 0.08, 0.12, 0.56]"
9,679075,2,30134,178,0.59,"[0.02, 0.04, 0.06, 0.08, 0.12]"
56,713910,1,29072,175,0.60,"[0.02, 0.04, 0.06, 0.08, 0.12, 0.54]"
